## Screen OTAVA LIBRARY

In [1]:
from rdkit import Chem
import os

sdf_path = r"D:\JK\CheB\OTAVA_Drug-Like_Green_Collection.sdf"

print(f"File size: {os.path.getsize(sdf_path)/1024/1024:.1f} MB")
print("Loading SDF file...")

supplier = Chem.SDMolSupplier(sdf_path)
total = len(supplier)
print(f"Total molecules: {total:,}")

# Preview first molecule properties
mol = supplier[0]
if mol is not None:
    props = mol.GetPropsAsDict()
    print(f"\nAvailable properties:")
    for key, val in list(props.items())[:15]:
        print(f"  {key:30s}: {val}")
    print(f"\nSMILES: {Chem.MolToSmiles(mol)[:60]}...")

File size: 420.9 MB
Loading SDF file...
Total molecules: 160,812

Available properties:
  Code                          : 129520004
  Number of H-Donors            : 2
  Number of H-Acceptors         : 8
  Number of Rotatable bonds     : 9
  CLogP                         : 2.5
  nRings                        : 3

SMILES: CCc1nnc(NS(=O)(=O)c2ccc(NC(=O)c3cc(OC)c(OC)c(OC)c3)cc2)s1...


In [2]:
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.Chem import rdMolDescriptors
import pandas as pd
import numpy as np
import time
import pickle
from collections import Counter

sdf_path = r"D:\JK\CheB\OTAVA_Drug-Like_Green_Collection.sdf"

# ZBG SMARTS patterns
zbg_patterns = {
    'Hydroxamic acid' : Chem.MolFromSmarts('[CX3](=O)[NX3][OH]'),
    'Carboxylic acid' : Chem.MolFromSmarts('[CX3](=O)[OH]'),
    'Thiol'           : Chem.MolFromSmarts('[SX2H]'),
    'Phosphonate'     : Chem.MolFromSmarts('[PX4](=O)([OH])[OH]'),
    'Sulfonamide'     : Chem.MolFromSmarts('[SX4](=O)(=O)[NX3]'),
    'Catechol'        : Chem.MolFromSmarts('c1ccc(O)c(O)c1'),
    'N-hydroxy'       : Chem.MolFromSmarts('[NX3][OH]'),
    'Hydroxypyridine' : Chem.MolFromSmarts('n1ccccc1O'),
    'Beta-lactam'     : Chem.MolFromSmarts('[NX3]1[CX4][CX3](=O)1'),
}

print("Step 1 — Filtering OTAVA library...")
print(f"Total compounds: 160,812")
print("Applying ZBG filter...\n")

supplier = Chem.SDMolSupplier(sdf_path)

filtered_data = []
processed = 0
passed_zbg = 0
t0 = time.time()

for i, mol in enumerate(supplier):
    if mol is None:
        continue

    processed += 1

    try:
        smiles = Chem.MolToSmiles(mol)

        # Get identifier
        try:
            identifier = mol.GetProp('Code')
        except:
            identifier = f'OTAVA_{i}'

        # Get existing properties
        try:
            mw   = Descriptors.MolWt(mol)
            logp = float(mol.GetProp('CLogP'))
            hbd  = int(mol.GetProp('Number of H-Donors'))
            hba  = int(mol.GetProp('Number of H-Acceptors'))
        except:
            mw   = Descriptors.MolWt(mol)
            logp = Descriptors.MolLogP(mol)
            hbd  = rdMolDescriptors.CalcNumHBD(mol)
            hba  = rdMolDescriptors.CalcNumHBA(mol)

        # ZBG filter
        found_zbgs = []
        for zbg_name, pattern in zbg_patterns.items():
            if pattern and mol.HasSubstructMatch(pattern):
                found_zbgs.append(zbg_name)

        if not found_zbgs:
            continue

        passed_zbg += 1
        filtered_data.append({
            'id'    : identifier,
            'smiles': smiles,
            'MW'    : round(mw, 2),
            'LogP'  : round(logp, 2),
            'HBD'   : hbd,
            'HBA'   : hba,
            'ZBG'   : ', '.join(found_zbgs)
        })

    except:
        continue

    if (i+1) % 20000 == 0:
        elapsed = round((time.time()-t0)/60, 1)
        print(f"  Processed: {i+1:,} | "
              f"ZBG pass: {passed_zbg:,} | "
              f"Time: {elapsed} mins")

# Save filtered
filtered_df = pd.DataFrame(filtered_data)
filtered_df.to_csv(
    project_path + r'\otava_filtered.csv', index=False
)

elapsed = round((time.time()-t0)/60, 1)
print(f"\n{'='*55}")
print(f"OTAVA FILTERING COMPLETE")
print(f"{'='*55}")
print(f"Total processed  : {processed:,}")
print(f"ZBG passed       : {passed_zbg:,} ({round(passed_zbg/processed*100,1)}%)")
print(f"Time taken       : {elapsed} mins")
print(f"\nZBG breakdown:")
all_zbgs = []
for zbg in filtered_df['ZBG']:
    all_zbgs.extend(zbg.split(', '))
for zbg, count in Counter(all_zbgs).most_common():
    print(f"  {zbg:25s}: {count:,}")

Step 1 — Filtering OTAVA library...
Total compounds: 160,812
Applying ZBG filter...

  Processed: 20,000 | ZBG pass: 4,086 | Time: 0.4 mins


[12:06:54]  deprecated group abbreviation ignored on line 9063173


  Processed: 120,000 | ZBG pass: 30,050 | Time: 2.5 mins


[12:07:44] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 1 ignored.
[12:07:44] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 1 ignored.
[12:07:45] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 2 ignored.
[12:07:45] Warning: ambiguous stereochemistry - opposing bonds have opposite wedging - at atom 2 ignored.


  Processed: 160,000 | ZBG pass: 38,344 | Time: 3.4 mins


NameError: name 'project_path' is not defined

In [3]:
project_path = r"C:\Users\DELL\Documents\SK\MyProject\Task1"

# Save filtered
filtered_df = pd.DataFrame(filtered_data)
filtered_df.to_csv(
    project_path + r'\otava_filtered.csv', index=False
)

from collections import Counter
elapsed = round((time.time()-t0)/60, 1)
print(f"\n{'='*55}")
print(f"OTAVA FILTERING COMPLETE")
print(f"{'='*55}")
print(f"Total processed  : {processed:,}")
print(f"ZBG passed       : {passed_zbg:,} ({round(passed_zbg/processed*100,1)}%)")
print(f"Time taken       : {elapsed} mins")
print(f"Saved to         : {project_path}\\otava_filtered.csv")
print(f"\nZBG breakdown:")
all_zbgs = []
for zbg in filtered_df['ZBG']:
    all_zbgs.extend(zbg.split(', '))
for zbg, count in Counter(all_zbgs).most_common():
    print(f"  {zbg:25s}: {count:,}")


OTAVA FILTERING COMPLETE
Total processed  : 160,812
ZBG passed       : 38,486 (23.9%)
Time taken       : 5.8 mins
Saved to         : C:\Users\DELL\Documents\SK\MyProject\Task1\otava_filtered.csv

ZBG breakdown:
  Sulfonamide              : 15,652
  Catechol                 : 13,806
  Carboxylic acid          : 10,524
  Thiol                    : 386
  Hydroxypyridine          : 309
  N-hydroxy                : 27
  Hydroxamic acid          : 5


In [4]:
import pandas as pd
import numpy as np
import pickle
import time
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.ML.Descriptors import MoleculeDescriptors

# Load models
with open(project_path + r'\models\rf_classifier.pkl', 'rb') as f:
    rf_model = pickle.load(f)
with open(project_path + r'\models\rf_regressor.pkl', 'rb') as f:
    rf_reg = pickle.load(f)
with open(project_path + r'\models\clean_descriptors.pkl', 'rb') as f:
    clean_descriptors = pickle.load(f)
with open(project_path + r'\models\descriptor_calculator.pkl', 'rb') as f:
    calculator = pickle.load(f)

descriptor_names = [d[0] for d in Descriptors.descList]
print("Models loaded ✅")

# Load filtered OTAVA
otava = pd.read_csv(project_path + r'\otava_filtered.csv')
print(f"Total ZBG compounds: {len(otava):,}")

def screen_compounds(compounds_df, label):
    print(f"\nScreening {label} ({len(compounds_df):,} compounds)...")
    t0 = time.time()

    desc_list, fp_list, valid_idx = [], [], []

    for i, smiles in enumerate(compounds_df['smiles']):
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is not None:
            descs = calculator.CalcDescriptors(mol)
            fp = AllChem.GetMorganFingerprintAsBitVect(
                mol, radius=2, nBits=2048)
            desc_list.append(descs)
            fp_list.append(list(fp))
            valid_idx.append(i)
        if (i+1) % 2000 == 0:
            elapsed = round((time.time()-t0)/60, 1)
            print(f"  {i+1:,}/{len(compounds_df):,} "
                  f"({elapsed} mins)...", end='\r')

    desc_df = pd.DataFrame(desc_list, columns=descriptor_names)
    fp_df   = pd.DataFrame(
        fp_list,
        columns=[f'Morgan_{i}' for i in range(2048)]
    )

    X_desc = desc_df[clean_descriptors].copy()
    X_desc = X_desc.replace([np.inf, -np.inf], np.nan)
    X_desc = X_desc.fillna(X_desc.median())
    X_desc = X_desc.clip(
        -np.finfo(np.float32).max,
         np.finfo(np.float32).max
    ).astype(np.float32)

    X_comb = pd.concat([
        X_desc.reset_index(drop=True),
        fp_df.reset_index(drop=True)
    ], axis=1)
    X_comb = X_comb.replace([np.inf, -np.inf], np.nan)
    X_comb = X_comb.fillna(X_comb.median())
    X_comb = X_comb.clip(
        -np.finfo(np.float32).max,
         np.finfo(np.float32).max
    ).astype(np.float32)

    activity_pred = rf_model.predict(X_desc)
    activity_prob = rf_model.predict_proba(X_desc)[:, 1]
    pic50_pred    = rf_reg.predict(X_comb)

    results = compounds_df.iloc[
        valid_idx
    ].reset_index(drop=True).copy()
    results['predicted_activity'] = activity_pred
    results['active_probability'] = activity_prob.round(4)
    results['predicted_pIC50']    = pic50_pred.round(3)

    actives = results[
        results['predicted_activity'] == 1
    ].sort_values(
        'active_probability', ascending=False
    ).reset_index(drop=True)

    elapsed = round((time.time()-t0)/60, 2)
    hit_rate = round(len(actives)/len(valid_idx)*100, 1)
    print(f"\nValid    : {len(valid_idx):,}")
    print(f"Actives  : {len(actives):,}")
    print(f"Hit rate : {hit_rate}%")
    print(f"Time     : {elapsed} mins")
    return actives


# Screen by priority
# Priority 1 — Hydroxamic acid (only 5!)
ha = otava[
    otava['ZBG'].str.contains('Hydroxamic acid', na=False)
].copy().reset_index(drop=True)
ha_hits = screen_compounds(ha, "Hydroxamic acid")
ha_hits.to_csv(
    project_path + r'\otava_ha_hits.csv', index=False
)

# Priority 2 — N-hydroxy + Thiol + Hydroxypyridine
p2 = otava[
    (
        otava['ZBG'].str.contains('N-hydroxy', na=False) |
        otava['ZBG'].str.contains('Thiol', na=False) |
        otava['ZBG'].str.contains('Hydroxypyridine', na=False)
    ) &
    (~otava['ZBG'].str.contains('Hydroxamic acid', na=False))
].copy().reset_index(drop=True)
p2_hits = screen_compounds(p2, "N-hydroxy + Thiol + Hydroxypyridine")
p2_hits.to_csv(
    project_path + r'\otava_p2_hits.csv', index=False
)

# Priority 3 — Sulfonamide
p3 = otava[
    otava['ZBG'].str.contains('Sulfonamide', na=False) &
    ~otava['ZBG'].str.contains('Hydroxamic acid', na=False) &
    ~otava['ZBG'].str.contains('N-hydroxy', na=False) &
    ~otava['ZBG'].str.contains('Thiol', na=False)
].copy().reset_index(drop=True)
p3_hits = screen_compounds(p3, "Sulfonamide")
p3_hits.to_csv(
    project_path + r'\otava_p3_hits.csv', index=False
)

# Priority 4 — Carboxylic acid + Catechol
p4 = otava[
    (
        otava['ZBG'].str.contains('Carboxylic acid', na=False) |
        otava['ZBG'].str.contains('Catechol', na=False)
    ) &
    ~otava['ZBG'].str.contains('Hydroxamic acid', na=False) &
    ~otava['ZBG'].str.contains('N-hydroxy', na=False) &
    ~otava['ZBG'].str.contains('Thiol', na=False) &
    ~otava['ZBG'].str.contains('Sulfonamide', na=False)
].copy().reset_index(drop=True)
p4_hits = screen_compounds(p4, "Carboxylic acid + Catechol")
p4_hits.to_csv(
    project_path + r'\otava_p4_hits.csv', index=False
)

# Combined summary
all_hits = pd.concat(
    [ha_hits, p2_hits, p3_hits, p4_hits],
    ignore_index=True
).sort_values(
    'active_probability', ascending=False
).drop_duplicates(subset=['smiles']).reset_index(drop=True)

# High confidence hits
high_conf = all_hits[
    all_hits['active_probability'] >= 0.85
].copy()

high_conf.to_csv(
    project_path + r'\otava_high_confidence_hits.csv',
    index=False
)

print(f"\n{'='*55}")
print(f"OTAVA SCREENING COMPLETE")
print(f"{'='*55}")
print(f"Hydroxamic acid hits : {len(ha_hits):,}")
print(f"N-hydroxy/Thiol hits : {len(p2_hits):,}")
print(f"Sulfonamide hits     : {len(p3_hits):,}")
print(f"Carboxylic/Catechol  : {len(p4_hits):,}")
print(f"Total unique hits    : {len(all_hits):,}")
print(f"High conf (≥0.85)    : {len(high_conf):,}")
print(f"\nTop 20 High Confidence Hits:")
print(high_conf.head(20)[[
    'id', 'ZBG', 'MW', 'LogP',
    'active_probability', 'predicted_pIC50'
]].to_string(index=False))

Models loaded ✅
Total ZBG compounds: 38,486

Screening Hydroxamic acid (5 compounds)...


[12:14:26] DEPRECATION WARNING: please use MorganGenerator
[12:14:26] DEPRECATION WARNING: please use MorganGenerator
[12:14:26] DEPRECATION WARNING: please use MorganGenerator
[12:14:26] DEPRECATION WARNING: please use MorganGenerator
[12:14:26] DEPRECATION WARNING: please use MorganGenerator



Valid    : 5
Actives  : 4
Hit rate : 80.0%
Time     : 0.18 mins

Screening N-hydroxy + Thiol + Hydroxypyridine (717 compounds)...


[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerator
[12:14:33] DEPRECATION WARNING: please use MorganGenerat


Valid    : 717
Actives  : 280
Hit rate : 39.1%
Time     : 0.5 mins

Screening Sulfonamide (15,651 compounds)...


...

  20,000/22,146 (13.0 mins)...

[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerator
[12:39:01] DEPRECATION WARNING: please use MorganGenerat

  22,000/22,146 (14.4 mins)...

[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerator
[12:40:22] DEPRECATION WARNING: please use MorganGenerat


Valid    : 22,146
Actives  : 14,362
Hit rate : 64.9%
Time     : 15.28 mins

OTAVA SCREENING COMPLETE
Hydroxamic acid hits : 4
N-hydroxy/Thiol hits : 280
Sulfonamide hits     : 12,228
Carboxylic/Catechol  : 14,362
Total unique hits    : 26,856
High conf (≥0.85)    : 108

Top 20 High Confidence Hits:
        id             ZBG     MW  LogP  active_probability  predicted_pIC50
   1097648 Carboxylic acid 466.49   2.8              0.9600            8.029
   1097649 Carboxylic acid 480.52   3.3              0.9400            8.028
   1098691 Carboxylic acid 444.48   2.7              0.9000            8.058
  11640456 Carboxylic acid 390.40   2.8              0.9000            8.096
   1157877 Carboxylic acid 455.90   2.7              0.9000            8.012
  15386412 Carboxylic acid 496.52   3.7              0.9000            8.323
7012160106     Sulfonamide 401.42   1.2              0.8941            8.268
   4357742        Catechol 492.53   2.5              0.8900            8.349
   109

In [5]:
import pandas as pd
import os

os.makedirs(project_path + r'\glide_docking', exist_ok=True)

# Load all OTAVA hit files
ha   = pd.read_csv(project_path + r'\otava_ha_hits.csv')
p2   = pd.read_csv(project_path + r'\otava_p2_hits.csv')
p3   = pd.read_csv(project_path + r'\otava_p3_hits.csv')
p4   = pd.read_csv(project_path + r'\otava_p4_hits.csv')

# ── Hydroxamic acid (only 4 — take all) ──
ha_dock = ha.sort_values(
    'active_probability', ascending=False
).reset_index(drop=True)

ha_dock[['id', 'smiles', 'ZBG', 'MW', 'LogP',
          'HBD', 'HBA', 'active_probability',
          'predicted_pIC50']].to_csv(
    project_path + r'\glide_docking\otava_ha_docking.csv',
    index=False
)
print(f"Hydroxamic acid CSV saved ✅ → {len(ha_dock)} compounds")

# ── N-hydroxy + Thiol ──
p2_dock = p2[
    p2['active_probability'] >= 0.85
].sort_values(
    'active_probability', ascending=False
).head(50).reset_index(drop=True)

p2_dock[['id', 'smiles', 'ZBG', 'MW', 'LogP',
          'HBD', 'HBA', 'active_probability',
          'predicted_pIC50']].to_csv(
    project_path + r'\glide_docking\otava_nhydroxy_thiol_docking.csv',
    index=False
)
print(f"N-hydroxy/Thiol CSV saved ✅ → {len(p2_dock)} compounds")

# ── Sulfonamide ──
p3_dock = p3[
    p3['active_probability'] >= 0.85
].sort_values(
    'active_probability', ascending=False
).head(50).reset_index(drop=True)

p3_dock[['id', 'smiles', 'ZBG', 'MW', 'LogP',
          'HBD', 'HBA', 'active_probability',
          'predicted_pIC50']].to_csv(
    project_path + r'\glide_docking\otava_sulfonamide_docking.csv',
    index=False
)
print(f"Sulfonamide CSV saved ✅ → {len(p3_dock)} compounds")

# ── Carboxylic acid + Catechol ──
p4_dock = p4[
    p4['active_probability'] >= 0.85
].sort_values(
    'active_probability', ascending=False
).head(50).reset_index(drop=True)

p4_dock[['id', 'smiles', 'ZBG', 'MW', 'LogP',
          'HBD', 'HBA', 'active_probability',
          'predicted_pIC50']].to_csv(
    project_path + r'\glide_docking\otava_carboxylic_catechol_docking.csv',
    index=False
)
print(f"Carboxylic/Catechol CSV saved ✅ → {len(p4_dock)} compounds")

# ── Combined all OTAVA high confidence ──
all_otava_dock = pd.concat(
    [ha_dock, p2_dock, p3_dock, p4_dock],
    ignore_index=True
).drop_duplicates(subset=['smiles']).sort_values(
    'active_probability', ascending=False
).reset_index(drop=True)

all_otava_dock[['id', 'smiles', 'ZBG', 'MW', 'LogP',
                 'HBD', 'HBA', 'active_probability',
                 'predicted_pIC50']].to_csv(
    project_path + r'\glide_docking\otava_all_docking.csv',
    index=False
)

print(f"\n{'='*55}")
print(f"ALL OTAVA DOCKING CSVs SAVED")
print(f"{'='*55}")
print(f"Hydroxamic acid     : {len(ha_dock)} compounds")
print(f"N-hydroxy/Thiol     : {len(p2_dock)} compounds")
print(f"Sulfonamide         : {len(p3_dock)} compounds")
print(f"Carboxylic/Catechol : {len(p4_dock)} compounds")
print(f"Combined total      : {len(all_otava_dock)} compounds")
print(f"\nFiles saved in glide_docking folder ✅")
print(f"\nTop 10 overall:")
print(all_otava_dock.head(10)[[
    'id', 'ZBG', 'MW', 'LogP',
    'active_probability', 'predicted_pIC50'
]].to_string(index=False))

Hydroxamic acid CSV saved ✅ → 4 compounds
N-hydroxy/Thiol CSV saved ✅ → 0 compounds
Sulfonamide CSV saved ✅ → 50 compounds
Carboxylic/Catechol CSV saved ✅ → 50 compounds

ALL OTAVA DOCKING CSVs SAVED
Hydroxamic acid     : 4 compounds
N-hydroxy/Thiol     : 0 compounds
Sulfonamide         : 50 compounds
Carboxylic/Catechol : 50 compounds
Combined total      : 104 compounds

Files saved in glide_docking folder ✅

Top 10 overall:
        id             ZBG     MW  LogP  active_probability  predicted_pIC50
   1097648 Carboxylic acid 466.49   2.8              0.9600            8.029
   1097649 Carboxylic acid 480.52   3.3              0.9400            8.028
  11640456 Carboxylic acid 390.40   2.8              0.9000            8.096
   1098691 Carboxylic acid 444.48   2.7              0.9000            8.058
  15386412 Carboxylic acid 496.52   3.7              0.9000            8.323
   1157877 Carboxylic acid 455.90   2.7              0.9000            8.012
7012160106     Sulfonamide 401.